# High Salary Employee Identification

## Problem Statement

The HR team wants to identify the highest-paid employees within each department.

For every department, return the top employees based on salary according to the ranking requirements provided.

## Input Tables

### he_department

| Column Name | Data Type |
|------------|-----------|
| department_id | INT |
| department_name | VARCHAR |

### he_employee

| Column Name | Data Type |
|------------|-----------|
| employee_id | INT |
| name | VARCHAR |
| salary | INT |
| department_id | INT |
| manager_id | DECIMAL |

## Requirements

- Consider employees assigned to a department.
- Rank employees within each department.
- Order employees by salary from highest to lowest.
- When salaries are equal, order employee names alphabetically.
- Keep the top three ranked positions per department.
- Employees with NULL department assignments should not be included.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| department_name |
| name |
| salary |

## Sample Input

### he_department

| department_id | department_name |
|--------------|-----------------|
| 1 | Data Analytics |
| 

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window

# he_department
he_department_schema = StructType([
    StructField("department_id", IntegerType(), True),
    StructField("department_name", StringType(), True)
])

he_department_data = [
    (1, "Data Analytics"),
    (2, "Data Science")
]

he_department_df = spark.createDataFrame(
    he_department_data,
    schema=he_department_schema
)

# he_employee
he_employee_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("department_id", IntegerType(), True),
    StructField("manager_id", DecimalType(10, 0), True)
])

he_employee_data = [
    (10, "James Anderson", 4000, 1, 11),
    (1, "Emma Thompson", 3800, 1, 6),
    (2, "Daniel Rodriguez", 2230, 1, 7),
    (3, "Olivia Smith", 2000, 1, 8),
    (4, "Noah Johnson", 6800, 2, 9),
    (8, "William Davis", 6800, 2, None)
]

he_employee_df = spark.createDataFrame(
    he_employee_data,
    ["employee_id", "name", "salary", "department_id", "manager_id"]
)

In [0]:
result_df = (
    he_employee_df.join(
        he_department_df, he_employee_df.department_id == he_department_df.department_id
    )
    .withColumn(
        "rank",
        dense_rank().over(
            Window.partitionBy("department_name").orderBy(desc("salary"), "name")
        ),
    )
    .select("department_name", "name", "salary")
    .filter(col("rank") <= 3)
    .orderBy("department_name")
)

display(result_df)